# 🚦 Первая нейронная сеть на Python

## Обучаем модель принимать решение по сигналам светофора

> Учебный практический модуль проекта **Almaz_AI**

В этом блокноте мы создадим простую нейронную сеть на **PyTorch**, которая получает состояние трёх ламп светофора и принимает решение:

- **0 — стоять**
- **1 — идти**

Главная цель — не автоматизация реального светофора, а понимание полного цикла обучения нейронной сети:

**данные → предсказание → ошибка → обратное распространение → изменение весов → новое предсказание**


---

# 1. Постановка задачи

У светофора три входных сигнала:

| Сигнал | Значение |
|---|---:|
| Красный | 0 или 1 |
| Жёлтый | 0 или 1 |
| Зелёный | 0 или 1 |

Результат работы модели:

| Выход модели | Решение |
|---:|---|
| 0 | Стоять |
| 1 | Идти |

В рамках учебного примера действует правило:

```text
Идти можно, когда зелёный включён и красный выключен.
```

Жёлтый не запрещает движение, если одновременно горит зелёный.

```text
GO = GREEN AND NOT RED
```


## Обычная программа и нейронная сеть

В обычной программе правило записывает программист:

```python
if green == 1 and red == 0:
    go
else:
    stop
```

В нейронной сети мы не прописываем это правило напрямую.

Мы показываем модели примеры правильных ответов:

```text
[0, 0, 1] → идти
[0, 1, 1] → идти
[1, 0, 1] → стоять
[0, 1, 0] → стоять
```

После этого сеть сама подбирает веса, которые позволяют воспроизводить требуемое поведение.


In [ ]:
def traffic_light_rule(red: int, yellow: int, green: int) -> int:
    """Возвращает решение по заранее заданному правилу светофора."""

    return int(green == 1 and red == 0)


examples = [
    (0, 0, 1),
    (0, 1, 1),
    (1, 0, 1),
    (0, 1, 0),
]

for red, yellow, green in examples:
    decision = traffic_light_rule(red, yellow, green)
    print(
        f"Красный={red}, Жёлтый={yellow}, Зелёный={green} "
        f"→ {'ИДТИ' if decision else 'СТОЯТЬ'}"
    )


---

# 2. Установка и импорт библиотек

Для работы понадобятся:

- `torch` — создание и обучение нейронной сети;
- `matplotlib` — построение графиков;
- `pandas` — красивое отображение таблиц;
- `numpy` — вспомогательные числовые операции.

В JupyterLab библиотеки можно установить следующей командой:

```python
%pip install torch matplotlib pandas numpy
```

Если они уже установлены, этот шаг можно пропустить.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

torch.manual_seed(42)

print("PyTorch version:", torch.__version__)


---

# 3. Полный набор состояний светофора

У трёх бинарных сигналов всего:

```text
2³ = 8 комбинаций
```

Мы используем все восемь комбинаций, поэтому сеть увидит полный набор возможных состояний.


In [ ]:
def create_traffic_light_dataset() -> tuple[torch.Tensor, torch.Tensor]:
    """Создаёт полный набор состояний светофора и правильных решений."""

    features = torch.tensor(
        [
            [0.0, 0.0, 0.0],
            [0.0, 0.0, 1.0],
            [0.0, 1.0, 0.0],
            [0.0, 1.0, 1.0],
            [1.0, 0.0, 0.0],
            [1.0, 0.0, 1.0],
            [1.0, 1.0, 0.0],
            [1.0, 1.0, 1.0],
        ],
        dtype=torch.float32,
    )

    targets = torch.tensor(
        [
            [0.0],
            [1.0],
            [0.0],
            [1.0],
            [0.0],
            [0.0],
            [0.0],
            [0.0],
        ],
        dtype=torch.float32,
    )

    return features, targets


features, targets = create_traffic_light_dataset()

dataset_table = pd.DataFrame(
    {
        "Красный": features[:, 0].int().numpy(),
        "Жёлтый": features[:, 1].int().numpy(),
        "Зелёный": features[:, 2].int().numpy(),
        "Целевой ответ": [
            "ИДТИ" if value == 1 else "СТОЯТЬ"
            for value in targets.squeeze().int().numpy()
        ],
    }
)

dataset_table


---

# 4. Визуализация всех состояний

На рисунке ниже каждая строка — отдельное состояние светофора.

Красный имеет высший приоритет:

- если красный включён — стоять;
- если красный выключен и зелёный включён — идти;
- иначе — стоять.


In [ ]:
def plot_traffic_light_states(
    features: torch.Tensor,
    targets: torch.Tensor,
) -> None:
    """Визуализирует все состояния светофора и правильные решения."""

    figure, axes = plt.subplots(
        nrows=2,
        ncols=4,
        figsize=(14, 7),
    )

    lamp_positions = [2, 1, 0]
    lamp_names = ["R", "Y", "G"]

    for index, axis in enumerate(axes.flat):
        red, yellow, green = features[index].int().tolist()
        values = [red, yellow, green]

        axis.set_xlim(-1, 1)
        axis.set_ylim(-0.8, 2.8)
        axis.set_aspect("equal")
        axis.axis("off")

        for position, label, value in zip(
            lamp_positions,
            lamp_names,
            values,
            strict=True,
        ):
            circle = plt.Circle(
                (0, position),
                0.28,
                alpha=1.0 if value else 0.15,
            )
            axis.add_patch(circle)
            axis.text(
                0.45,
                position,
                label,
                va="center",
                fontsize=10,
            )

        action = "ИДТИ" if targets[index].item() == 1 else "СТОЯТЬ"
        axis.set_title(
            f"{red}{yellow}{green} → {action}",
            fontsize=11,
        )

    figure.suptitle(
        "Все возможные состояния светофора",
        fontsize=16,
    )
    plt.tight_layout()
    plt.show()


plot_traffic_light_states(features, targets)


---

# 5. Архитектура нейронной сети

Наша модель будет иметь следующую структуру:

```text
Красный ──┐
          │
Жёлтый ───┼──► 4 скрытых нейрона ──► 1 выход
          │
Зелёный ──┘
```

Состав модели:

1. Входной слой — 3 сигнала.
2. Скрытый слой — 4 нейрона.
3. Функция активации `ReLU`.
4. Выходной слой — 1 число.
5. `Sigmoid` преобразует выход в вероятность движения.


In [ ]:
class TrafficLightNetwork(nn.Module):
    """Нейронная сеть, принимающая решение по состоянию светофора."""

    def __init__(self) -> None:
        """Создаёт входной, скрытый и выходной слои."""

        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(3, 4),
            nn.ReLU(),
            nn.Linear(4, 1),
        )

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        """Выполняет прямой проход и возвращает логиты модели."""

        return self.network(features)


model = TrafficLightNetwork()
model


---

# 6. Предсказания до обучения

До обучения веса модели случайные.

Поэтому ответы сети могут быть неправильными и не иметь логики. Это нормальное начальное состояние.


In [ ]:
def get_predictions(
    model: nn.Module,
    features: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Возвращает вероятности и бинарные решения модели."""

    model.eval()

    with torch.no_grad():
        logits = model(features)
        probabilities = torch.sigmoid(logits)
        predictions = (probabilities >= 0.5).int()

    return probabilities, predictions


def build_prediction_table(
    model: nn.Module,
    features: torch.Tensor,
    targets: torch.Tensor,
) -> pd.DataFrame:
    """Формирует таблицу предсказаний модели."""

    probabilities, predictions = get_predictions(model, features)

    return pd.DataFrame(
        {
            "Красный": features[:, 0].int().numpy(),
            "Жёлтый": features[:, 1].int().numpy(),
            "Зелёный": features[:, 2].int().numpy(),
            "Вероятность идти": probabilities.squeeze().numpy(),
            "Ответ сети": [
                "ИДТИ" if value == 1 else "СТОЯТЬ"
                for value in predictions.squeeze().numpy()
            ],
            "Правильный ответ": [
                "ИДТИ" if value == 1 else "СТОЯТЬ"
                for value in targets.squeeze().int().numpy()
            ],
        }
    )


prediction_table_before = build_prediction_table(
    model,
    features,
    targets,
)

prediction_table_before


---

# 7. Как проходит обучение

Каждая эпоха состоит из четырёх основных операций:

```python
logits = model(features)
loss = loss_function(logits, targets)

optimizer.zero_grad()
loss.backward()
optimizer.step()
```

Соответствие теории и кода:

| Теория | Python |
|---|---|
| Прямой проход | `model(features)` |
| Вычисление ошибки | `loss_function(...)` |
| Обратное распространение | `loss.backward()` |
| Изменение весов | `optimizer.step()` |

Мы используем:

- `BCEWithLogitsLoss` — функцию потерь для бинарного решения;
- `Adam` — алгоритм изменения весов.


In [ ]:
def train_traffic_light_network(
    model: nn.Module,
    features: torch.Tensor,
    targets: torch.Tensor,
    epochs: int = 1000,
    learning_rate: float = 0.05,
) -> list[float]:
    """Обучает нейросеть принимать решение по сигналам светофора."""

    loss_function = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
    )

    loss_history: list[float] = []

    for epoch in range(epochs):
        model.train()

        logits = model(features)
        loss = loss_function(logits, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        loss_history.append(float(loss.item()))

        if epoch % 100 == 0:
            print(
                f"Эпоха {epoch:04d} | "
                f"Ошибка: {loss.item():.6f}"
            )

    return loss_history


loss_history = train_traffic_light_network(
    model=model,
    features=features,
    targets=targets,
)


---

# 8. График уменьшения ошибки

Если обучение проходит правильно, `Loss` постепенно уменьшается.

Это означает, что предсказания модели приближаются к правильным ответам.


In [ ]:
def plot_loss_history(loss_history: list[float]) -> None:
    """Показывает изменение ошибки нейросети во время обучения."""

    plt.figure(figsize=(10, 5))
    plt.plot(loss_history)
    plt.title("Обучение нейросети светофора")
    plt.xlabel("Эпоха")
    plt.ylabel("Ошибка Loss")
    plt.grid(True)
    plt.show()


plot_loss_history(loss_history)


---

# 9. Предсказания после обучения

Теперь сеть должна правильно распознавать все восемь состояний.


In [ ]:
prediction_table_after = build_prediction_table(
    model,
    features,
    targets,
)

prediction_table_after


In [ ]:
def calculate_accuracy(
    model: nn.Module,
    features: torch.Tensor,
    targets: torch.Tensor,
) -> float:
    """Вычисляет точность решений нейронной сети."""

    _, predictions = get_predictions(model, features)
    correct = predictions.float() == targets
    accuracy = correct.float().mean().item()

    return float(accuracy)


accuracy = calculate_accuracy(
    model,
    features,
    targets,
)

print(f"Точность модели: {accuracy:.2%}")


---

# 10. Сравнение до и после обучения

До обучения:

- веса случайные;
- решения могут быть неверными;
- вероятность движения не отражает правило.

После обучения:

- только зелёный → идти;
- жёлтый и зелёный → идти;
- любой красный сигнал → стоять.


In [ ]:
comparison = pd.DataFrame(
    {
        "Состояние": [
            f"{int(r)}{int(y)}{int(g)}"
            for r, y, g in features.tolist()
        ],
        "До обучения": prediction_table_before["Ответ сети"],
        "После обучения": prediction_table_after["Ответ сети"],
        "Правильный ответ": prediction_table_after["Правильный ответ"],
    }
)

comparison


---

# 11. Что именно выучила сеть?

Модель не хранит внутри текстовое правило:

```text
Если зелёный включён и красный выключен — идти.
```

Она хранит числовые веса.

Эти веса преобразуют входы:

```text
[красный, жёлтый, зелёный]
```

в выходную вероятность:

```text
0.03 → стоять
0.98 → идти
```

Таким образом, обучение — это поиск подходящих числовых параметров.


In [ ]:
def show_model_parameters(model: nn.Module) -> None:
    """Выводит обученные веса и смещения модели."""

    for name, parameter in model.named_parameters():
        print(f"\n{name}")
        print(parameter.detach())


show_model_parameters(model)


---

# 12. Важное практическое замечание

Для реального светофора нейронная сеть здесь не нужна.

Строгое правило безопаснее записать обычным условием:

```python
return green == 1 and red == 0
```

В этой главе нейронная сеть используется как **учебный стенд**, потому что пример позволяет наглядно изучить:

- входные признаки;
- целевые ответы;
- слои и нейроны;
- функцию активации;
- прямой проход;
- функцию потерь;
- backpropagation;
- оптимизатор;
- изменение весов;
- итоговое предсказание.


---

# 13. Что нужно запомнить

- Нейросеть обучается на примерах.
- Каждый сигнал светофора является входным признаком.
- Правильное действие является целевым ответом.
- До обучения веса случайные.
- `Loss` показывает величину ошибки.
- `loss.backward()` вычисляет направление изменения весов.
- `optimizer.step()` обновляет веса.
- После обучения сеть воспроизводит нужное правило.


---

# 14. Вопросы для самопроверки

1. Сколько входов получает модель?
2. Почему существует ровно восемь состояний светофора?
3. Что означает выход `0`?
4. Что означает выход `1`?
5. Что происходит во время прямого прохода?
6. Что показывает `Loss`?
7. Что делает `loss.backward()`?
8. Для чего нужен оптимизатор?
9. Почему до обучения ответы случайные?
10. Почему для реального светофора лучше обычное условие?


---

# 15. Самостоятельные задания

## Задание 1

Измените правило:

```text
На жёлтый и зелёный одновременно нужно стоять.
```

Обновите целевые ответы и повторно обучите модель.

## Задание 2

Добавьте четвёртый вход:

```text
Пешеход нажал кнопку перехода.
```

Разрешайте движение только при включённой кнопке.

## Задание 3

Измените количество скрытых нейронов:

```python
nn.Linear(3, 2)
nn.Linear(3, 8)
nn.Linear(3, 16)
```

Сравните скорость обучения.

## Задание 4

Замените оптимизатор `Adam` на `SGD` и сравните графики `Loss`.

## Задание 5

Сохраните обученную модель:

```python
torch.save(model.state_dict(), "traffic_light_model.pt")
```

Затем загрузите её в новом сеансе.


---

# 16. Следующий учебный модуль

После этого блокнота логично изучить:

1. функции активации `ReLU`, `Sigmoid`, `Tanh`;
2. функцию потерь;
3. градиентный спуск;
4. обучение сети на неполных данных;
5. разделение на `train` и `test`;
6. классификацию изображений;
7. основы Transformer;
8. локальные языковые модели;
9. RAG;
10. агентов и инструменты;
11. fine-tuning.

---

## Итог

Мы создали первую полноценную нейронную сеть, которая:

- получает три сигнала;
- обучается на примерах;
- самостоятельно изменяет веса;
- уменьшает ошибку;
- принимает правильное решение по светофору.

Это первый практический шаг от общей теории нейронных сетей к PyTorch, Transformer, локальным моделям и fine-tuning.
